# Validate and inspect synthetic question pairs

**Audience:** Python developers preparing reproducible data workflows. **Time:** about 15–25 minutes. **Execution:** offline; no credentials, paid APIs, or downloaded datasets.

Validate a question-pair dataset, detect conflicting duplicates, assign leakage-resistant component splits, and display escaped review cards.

## Prerequisites and setup

Use the tested Python environment and exact dependency versions in the [collection setup](../../../README.md). Install from the collection root before opening this notebook, then select that environment's kernel. This notebook uses Python standard library and pandas. Restart the kernel and run all cells in order. No working-directory assumptions or private files are required.

## Data provenance

All examples below are newly authored synthetic fixtures. Names, identifiers, coordinates and values are invented for this tutorial and do not represent real people, accounts, organizations or operational records. Code is newly authored for this example; the references explain the underlying APIs.

Copyright IBM Corp. 2026

SPDX-License-Identifier: Apache-2.0

**License and AI-assistance notice.** Source code is licensed under Apache License 2.0. Modified for open-source preparation on 2026-10-23. This notebook may include contributions involving Anthropic Claude Opus 4+, Claude Sonnet 3.7+, and OpenAI GPT-4+ models; exact per-file model provenance is unverified. See [AI disclosure](../../../AI_DISCLOSURE.md), [license](../../../LICENSE), and [attribution](../../../NOTICE.md). Human contributors remain responsible for review, rights and DCO certification.


## 1. Define a small labeled dataset

The label records an author's intended meaning relation: `1` means equivalent, `0` means different. It is not a model prediction. Shared questions must remain in the same split or evaluation can leak through a rephrased duplicate.

In [ ]:
import re
from collections import defaultdict
from hashlib import sha256
from html import escape

import pandas as pd
from IPython.display import HTML, display

pairs = [
    {
        "pair_id": "pair-1",
        "left": "How do I reset a timer?",
        "right": "How can a timer be reset?",
        "label": 1,
    },
    {
        "pair_id": "pair-2",
        "left": "How can a timer be reset?",
        "right": "How do I start a timer?",
        "label": 0,
    },
    {
        "pair_id": "pair-3",
        "left": "Where is the settings menu?",
        "right": "How do I open settings?",
        "label": 1,
    },
    {
        "pair_id": "pair-4",
        "left": "Can I export a report?",
        "right": "Can I delete a report?",
        "label": 0,
    },
    {
        "pair_id": "pair-5",
        "left": "How do I rename a project?",
        "right": "Can a project name be changed?",
        "label": 1,
    },
]
pd.DataFrame(pairs)

## 2. Normalize for duplicate detection

Whitespace and case normalization are conservative. Removing punctuation or stemming could accidentally conflate distinct questions. Pair order does not change equivalence, so duplicate keys use a sorted pair.

Require each pair identifier to be nonblank text before duplicate detection or rendering. An identifier that happens to be hashable, such as a number or `None`, is not a valid display identifier.

In [ ]:
def normalize(text):
    if not isinstance(text, str) or not text.strip():
        raise ValueError("Question must be nonempty text")
    return re.sub(r"\s+", " ", text).strip().casefold()


def validate_pairs(records):
    seen_ids, labels = set(), {}
    for row in records:
        identifier = row.get("pair_id")
        if not isinstance(identifier, str) or not identifier.strip():
            raise ValueError("Pair identifier must be nonblank text")
        if identifier in seen_ids:
            raise ValueError("Duplicate pair identifier")
        seen_ids.add(identifier)
        if type(row["label"]) is not int or row["label"] not in (0, 1):
            raise ValueError("Label must be integer 0 or 1")
        key = tuple(sorted((normalize(row["left"]), normalize(row["right"]))))
        if key in labels and labels[key] != row["label"]:
            raise ValueError("Conflicting labels for the same pair")
        if key[0] == key[1] and row["label"] == 0:
            raise ValueError("Identical normalized questions cannot have label 0")
        labels[key] = row["label"]
    return True


assert validate_pairs(pairs)

## 3. Split connected components, not independent rows

Construct an undirected graph where any paired questions are connected, regardless of their label. All nodes in a connected component go to one split. A stable digest makes the assignment independent of input ordering; it does not guarantee balanced splits for a tiny dataset.

This authored fixture includes a disconnected component that hashes to validation, so both splits are exercised. Checking only that questions never cross splits would also pass a broken implementation that assigned every row to training.

In [ ]:
def assign_splits(records):
    validate_pairs(records)
    graph = defaultdict(set)
    for row in records:
        a, b = normalize(row["left"]), normalize(row["right"])
        graph[a].add(b)
        graph[b].add(a)
    split_for = {}
    for start in sorted(graph):
        if start in split_for:
            continue
        pending, component = [start], set()
        while pending:
            node = pending.pop()
            if node not in component:
                component.add(node)
                pending.extend(graph[node] - component)
        digest = sha256("\0".join(sorted(component)).encode()).hexdigest()
        split = "validation" if int(digest[:8], 16) % 5 == 0 else "train"
        split_for.update({node: split for node in component})
    return [{**row, "split": split_for[normalize(row["left"])]} for row in records]


review = assign_splits(pairs)
assert {row["split"] for row in review} == {"train", "validation"}
assert next(row["split"] for row in review if row["pair_id"] == "pair-5") == "validation"
question_splits = defaultdict(set)
for row in review:
    for field in ("left", "right"):
        question_splits[normalize(row[field])].add(row["split"])
assert all(len(splits) == 1 for splits in question_splits.values())
assert {r["pair_id"]: r["split"] for r in review} == {
    r["pair_id"]: r["split"] for r in assign_splits(list(reversed(pairs)))
}
pd.DataFrame(review)

## 4. Render escaped review cards

Treat text as data even inside an HTML notebook output. Escaping prevents a question containing markup from becoming executable markup. The data frame remains available for readers who prefer plain tabular output.

In [ ]:
def review_card(row):
    return (
        "<section><h4>"
        + escape(row["pair_id"])
        + "</h4><p>"
        + escape(row["left"])
        + "</p><p>"
        + escape(row["right"])
        + "</p><p>Equivalent label: "
        + str(row["label"])
        + "</p></section>"
    )


for row in review[:2]:
    display(HTML(review_card(row)))
probe = dict(review[0], left="<script>alert(1)</script>")
assert "<script>" not in review_card(probe)
assert "&lt;script&gt;" in review_card(probe)
try:
    validate_pairs(pairs + [dict(pairs[0], pair_id="pair-6", label=0)])
except ValueError:
    print("Conflicting duplicate rejected.")
else:
    raise AssertionError("Conflicting label accepted")
# Missing, null, numeric and blank identifiers must fail before HTML rendering.
invalid_identifiers = [dict(pairs[0], pair_id=value) for value in (None, 123, "", "   ")]
invalid_identifiers.append({key: value for key, value in pairs[0].items() if key != "pair_id"})
for invalid in invalid_identifiers:
    try:
        validate_pairs([invalid])
    except ValueError:
        pass
    else:
        raise AssertionError("Invalid pair identifier accepted")
print("Five identifier rejection cases passed; both splits are represented.")

## Validation and interpretation

The assertions above are executable acceptance criteria. A failed assertion is a failed example: inspect the inputs and kernel versions before continuing. Inspect the displayed tables or figures as well; successful execution alone does not establish fitness for a different dataset.

## Limitations and extensions

Five pairs are far too few for a benchmark, and the labels have no independent annotation study. Components can be very large, leading to imbalanced splits. Real evaluation needs domain-specific annotation guidelines, adjudication, representative sampling and held-out data collected independently of development.

## Cleanup

This notebook creates no persistent files, network sessions or cloud resources. Restart the kernel to discard in-memory objects. If you adapt it to real data, keep inputs outside a public repository and review every output before sharing.

## References and attribution

- [Python html.escape](https://docs.python.org/3/library/html.html#html.escape)
- [Python collections](https://docs.python.org/3/library/collections.html)

The example is prepared for editorial review; it is not an IBM endorsement or a claim of production certification.